# 03 — Özellik Seçimi

**Amaç:** 3 farklı özellik seçimi stratejisini karşılaştırmak:
1. Korelasyon analizi (yüksek korelasyonlu özellikleri tespit et)
2. Random Forest önem skorları (Top-20)
3. Mutual Information (SelectKBest, k=20)
4. Vajrobol'un 5 özelliği (literatür referansı)

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from src.preprocessing import full_preprocessing_pipeline
from src.feature_selection import (
    correlation_analysis, rf_feature_importance,
    mutual_info_selection, get_top5_vajrobol
)

DATA_PATH = '../data/PhiUSIIL_Phishing_URL_Dataset.csv'
FIG_DIR = '../results/figures/'
METRICS_DIR = '../results/metrics/'
FIG_KWARGS = dict(dpi=150, bbox_inches='tight')

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

sns.set_theme(style='whitegrid')

## 1. Ön İşlenmiş Veriyi Yükle

In [ ]:
X_train_sc, X_test_sc, y_train, y_test, tld_encoder, scaler = full_preprocessing_pipeline(DATA_PATH)

feature_names = list(X_train_sc.columns)
print(f'Özellik sayısı: {len(feature_names)}')

## 2. Korelasyon Analizi

In [ ]:
high_corr_features, corr_matrix = correlation_analysis(X_train_sc, threshold=0.9)
print(f'\nYüksek korelasyonlu (>0.9) özellik sayısı: {len(high_corr_features)}')

## 3. Random Forest Önem Skorları

In [ ]:
print('RF önem skorları hesaplanıyor (birkaç dakika sürebilir)...')
rf_top20, rf_importance_df = rf_feature_importance(X_train_sc, y_train, top_n=20)

fig, ax = plt.subplots(figsize=(10, 7))
top20_df = rf_importance_df.head(20)
ax.barh(range(20), top20_df['importance'].values[::-1])
ax.set_yticks(range(20))
ax.set_yticklabels(top20_df['feature'].values[::-1], fontsize=9)
ax.set_xlabel('Importance Score')
ax.set_title('RF Feature Importance — Top-20')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}rf_feature_importance.png', **FIG_KWARGS)
plt.show()

rf_importance_df.to_csv(f'{METRICS_DIR}rf_importance_scores.csv', index=False)
print('\nTop-20 RF özellikleri:')
print(rf_top20)

## 4. Mutual Information Seçimi

In [ ]:
print('Mutual Information hesaplanıyor...')
mi_top20, mi_scores = mutual_info_selection(X_train_sc, y_train, k=20)

fig, ax = plt.subplots(figsize=(10, 7))
mi_top20_df = mi_scores.head(20)
ax.barh(range(20), mi_top20_df['mi_score'].values[::-1])
ax.set_yticks(range(20))
ax.set_yticklabels(mi_top20_df['feature'].values[::-1], fontsize=9)
ax.set_xlabel('Mutual Information Score')
ax.set_title('Mutual Information — Top-20 Özellik')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}mutual_info_importance.png', **FIG_KWARGS)
plt.show()

mi_scores.to_csv(f'{METRICS_DIR}mutual_info_scores.csv', index=False)

## 5. Vajrobol'un 5 Özelliği

In [ ]:
vajrobol_5 = get_top5_vajrobol()
print('Vajrobol\'un 5 özelliği:', vajrobol_5)

available = [f for f in vajrobol_5 if f in feature_names]
missing = [f for f in vajrobol_5 if f not in feature_names]
print(f'Mevcut: {available}')
print(f'Eksik:  {missing}')

## 6. Özellik Seçim Yöntemlerinin Karşılaştırması

In [ ]:
rf_set = set(rf_top20)
mi_set = set(mi_top20)
vajrobol_set = set(available)

print('RF ∩ MI (ortak özellikler):',  rf_set & mi_set)
print('Vajrobol ⊂ RF Top-20:', vajrobol_set.issubset(rf_set))
print('Vajrobol ⊂ MI Top-20:', vajrobol_set.issubset(mi_set))

# Venn benzeri karşılaştırma tablosu
comparison = pd.DataFrame({
    'Feature': sorted(rf_set | mi_set | vajrobol_set),
})
comparison['RF Top-20'] = comparison['Feature'].isin(rf_set)
comparison['MI Top-20'] = comparison['Feature'].isin(mi_set)
comparison['Vajrobol'] = comparison['Feature'].isin(vajrobol_set)
print('\nÖzellik seçim yöntemi karşılaştırması:')
print(comparison.to_string(index=False))

comparison.to_csv(f'{METRICS_DIR}feature_selection_comparison.csv', index=False)

## 7. Özellik Setlerini Kaydet

In [ ]:
feature_sets = {
    'all_features': feature_names,
    'rf_top20': rf_top20,
    'mi_top20': mi_top20,
    'vajrobol_5': available,
}
joblib.dump(feature_sets, f'{METRICS_DIR}feature_sets.joblib')
print('Özellik setleri kaydedildi: results/metrics/feature_sets.joblib')

print('\nÖzet:')
for name, feats in feature_sets.items():
    print(f'  {name}: {len(feats)} özellik')